In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
df = pd.read_csv("train.csv")

print("Columns:", df.columns)
df.head()

Columns: Index(['id', 'subject', 'body', 'text', 'category', 'category_id'], dtype='object')


,id,subject,body,text,category,category_id
0,promotions_582,Anniversary Special: Buy one get one free,"As our loyal customer, get exclusive $60 off $...",Anniversary Special: Buy one get one free As o...,promotions,1
1,spam_1629,Your Amazon was used on new device,Your $5000 refund is processed. Claim: bit.ly/...,Your Amazon was used on new device Your $5000 ...,spam,3
2,spam_322,Re: Your Google inquiry,"Hi, following up about your Google application...","Re: Your Google inquiry Hi, following up about...",spam,3
3,social_media_80,Digital Ritual Experience Creation,Cross-cultural ceremony design. Join: virtualr...,Digital Ritual Experience Creation Cross-cultu...,social_media,2
4,forum_1351,"Your post was moved to ""Programming Help""","Trending: ""cooking"" (258 comments). View: supp...","Your post was moved to ""Programming Help"" Tren...",forum,0


In [4]:
if "text" not in df.columns:
    if "subject" in df.columns and "body" in df.columns:
        df["text"] = df["subject"].fillna('') + " " + df["body"].fillna('')
    else:
        raise ValueError("Dataset must contain text OR subject + body")

In [5]:
def clean_email(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_email)

In [6]:
def rule_based_urgency(text):
    text = text.lower()

    if any(word in text for word in ["urgent", "asap", "immediately", "not working", "failed"]):
        return "high"
    elif any(word in text for word in ["soon", "please check", "issue", "help"]):
        return "medium"
    else:
        return "low"

In [7]:
if "urgency" not in df.columns:
    print("Creating urgency column using rules...")
    df["urgency"] = df["clean_text"].apply(rule_based_urgency)

Creating urgency column using rules...


In [8]:
X = df["clean_text"]
y = df["urgency"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [10]:
def predict_urgency(text):
    cleaned = clean_email(text)
    vectorized = vectorizer.transform([cleaned])
    return model.predict(vectorized)[0]

print(predict_urgency("My system is not working please fix ASAP"))

low
